In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth.praat import call
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import pickle
from datetime import datetime
import shutil
import re
import random
warnings.filterwarnings('ignore')

# ==================== ДИАГНОСТИКА GPU ====================
print("="*80)
print("ДИАГНОСТИКА CUDA/GPU")
print("="*80)
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA версия: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CUDA НЕДОСТУПНА! Обучение будет на CPU")
print("="*80)

# ==================== ПАРАМЕТРЫ ====================
SEGMENT_DURATION = 7.0
SAMPLE_RATE = 16000
N_MELS = 128
HOP_LENGTH = 256
N_FFT = 1024
TARGET_FRAMES = int(SEGMENT_DURATION * SAMPLE_RATE / HOP_LENGTH)

# CutMix параметры
USE_CUTMIX = True
CUTMIX_PROB = 0.7
CUTMIX_BETA = 1.0

BATCH_SIZE = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 10000
EARLY_STOPPING_PATIENCE = 100
IMPROVEMENT_THRESHOLD = 0.001
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Кросс-валидация
N_SPLITS = 5

# 🆕 ПАРАМЕТРЫ ДЛЯ ЯЗЫКОВЫХ ГРУПП
NUM_LANGUAGES = 3
LANGUAGE_KEYWORDS = {
    0:['rus', 'russian', 'рус', 'ру'],
    1: ['tat', 'tatar', 'тат', 'тт'],
    2: ['bil', 'bilingual', 'би', 'билингв']
}
LANG_NAMES = {0: 'Русский', 1: 'Татарский', 2: 'Билингв'}

print(f"🎯 Устройство для обучения: {DEVICE}")
print(f"🎯 Количество фолдов кросс-валидации: {N_SPLITS}")
print(f"🎯 Языковых групп: {NUM_LANGUAGES}")

# ==================== ЗАГРУЗКА ДАННЫХ ====================
print("\nЗагрузка данных из папок...")
data_root = Path("C:\\Users\\erith\\Downloads\\Чтение текста\\Чтение текста")
if not data_root.exists():
    raise FileNotFoundError(f"Директория не найдена: {data_root}")

def detect_language_group_from_filename(filename):
    """Определение языковой группы по имени файла"""
    filename_lower = filename.lower()
    for lang_id, keywords in LANGUAGE_KEYWORDS.items():
        for keyword in keywords:
            if keyword in filename_lower:
                return lang_id
    return 0

df_records = []
patient_ids =[]
language_stats = {0: 0, 1: 0, 2: 0}

for label, folder_name in[("PD", "Болезнь Паркинсона_Parkinson's disease (PD)"), 
                           ("Control", "Контроль_Control (C)")]:
    folder = data_root / folder_name
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder}")
    
    wav_files = list(folder.glob("*.wav"))
    print(f"Найдено {len(wav_files)} файлов в папке: {folder_name}")
    
    for wav_file in wav_files:
        try:
            match = re.match(r'(\d+)(?:\(\d+\))?_(PD\d+|C)_(Male|Female)(?:_(nolevodopa|off|on))?_(\w+)\.wav$',
                           wav_file.name, re.I)
            patient_id = int(match.group(1)) if match else hash(wav_file.name) % 10000
            
            # 🆕 Определение языковой группы
            language_id = detect_language_group_from_filename(wav_file.name)
            language_stats[language_id] += 1
            
            sr = librosa.get_samplerate(wav_file)
            duration = librosa.get_duration(path=wav_file)
            
            df_records.append({
                "filepath": str(wav_file),
                "filename": wav_file.name,
                "label": label,
                "sr": sr,
                "duration": duration,
                "patient_id": patient_id,
                "language_id": language_id
            })
            patient_ids.append(patient_id)
        except Exception as e:
            print(f"⚠️ Ошибка при обработке {wav_file.name}: {e}")

df = pd.DataFrame(df_records)
print(f"\n✅ Загружено файлов: {len(df)}")
print(f"✅ Уникальных пациентов: {len(np.unique(patient_ids))}")
print(f"\n📊 Распределение по языковым группам:")
for lang_id, count in language_stats.items():
    print(f"   {LANG_NAMES[lang_id]}: {count} файлов ({count/len(df)*100:.1f}%)")

# ==================== ЭКСТРАКТОР ПРИЗНАКОВ (123 признака) ====================
def extract_acoustic_features_for_segment(segment, sr=SAMPLE_RATE):
    features =[]
    try:
        sound = parselmouth.Sound(segment, sampling_frequency=sr)
        
        # ===== MFCC (80 признаков) =====
        if hasattr(sound, 'to_mfcc'):
            mfcc = sound.to_mfcc(number_of_coefficients=13)
            if hasattr(mfcc, 'to_array'):
                mfcc_matrix = mfcc.to_array()
                for i in range(min(13, mfcc_matrix.shape[0])):
                    coef_values = mfcc_matrix[i, :]
                    if len(coef_values) > 0 and not np.all(np.isnan(coef_values)):
                        coef_values = coef_values[~np.isnan(coef_values)]
                        features.extend([
                            np.mean(coef_values), np.std(coef_values),
                            np.min(coef_values), np.max(coef_values),
                            np.max(coef_values) - np.min(coef_values),
                            np.median(coef_values)
                        ])
                    else:
                        features.extend([0.0] * 6)
                if mfcc_matrix.size > 0:
                    features.extend([np.mean(mfcc_matrix), np.std(mfcc_matrix)])
                else:
                    features.extend([0.0, 0.0])
            else:
                features.extend([0.0] * 80)
        else:
            features.extend([0.0] * 80)
        
        # ===== Pitch (6 признаков) =====
        pitch = call(sound, "To Pitch", 0.0, 75, 500)
        pitch_values = pitch.selected_array['frequency']
        pitch_values = pitch_values[pitch_values > 0]
        if len(pitch_values) > 0:
            features.extend([
                np.mean(pitch_values), np.std(pitch_values),
                np.min(pitch_values), np.max(pitch_values),
                np.max(pitch_values) - np.min(pitch_values),
                np.median(pitch_values)
            ])
        else:
            features.extend([0.0] * 6)
        
        # ===== Intensity (5 признаков) =====
        intensity = sound.to_intensity()
        intensity_values = intensity.values[0]
        intensity_values = intensity_values[~np.isnan(intensity_values)]
        if len(intensity_values) > 0:
            features.extend([
                np.mean(intensity_values), np.std(intensity_values),
                np.min(intensity_values), np.max(intensity_values),
                np.max(intensity_values) - np.min(intensity_values)
            ])
        else:
            features.extend([0.0] * 5)
        
        # ===== Formants F1-F3 (12 признаков) =====
        formants = sound.to_formant_burg()
        for formant_num in[1, 2, 3]:
            f_values =[]
            for t in formants.xs():
                try:
                    f_val = formants.get_value_at_time(formant_num, t)
                    if not np.isnan(f_val) and f_val > 0:
                        f_values.append(f_val)
                except:
                    continue
            if f_values:
                features.extend([
                    np.mean(f_values), np.std(f_values),
                    np.min(f_values), np.max(f_values)
                ])
            else:
                features.extend([0.0] * 4)
        
        # ===== Spectral features (9 признаков) =====
        spectrum = sound.to_spectrum()
        features.extend([
            spectrum.get_centre_of_gravity(),
            spectrum.get_standard_deviation(),
            spectrum.get_skewness(),
            spectrum.get_kurtosis()
        ])
        
        spectrum_values = spectrum.values[0]
        n = len(spectrum_values)
        max_freq = sr / 2
        frequencies = np.linspace(0, max_freq, n)
        bands =[(0, 250), (250, 500), (500, 1000), (1000, 2000), (2000, 4000)]
        total_energy = np.sum(spectrum_values ** 2) + 1e-8
        for low_freq, high_freq in bands:
            band_mask = (frequencies >= low_freq) & (frequencies < high_freq)
            if np.any(band_mask):
                band_energy = np.sum(spectrum_values[band_mask] ** 2)
                features.append(band_energy / total_energy)
            else:
                features.append(0.0)
        
        # ===== HNR (4 признака) =====
        hnr = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr_values = hnr.values[0]
        hnr_values = hnr_values[~np.isnan(hnr_values)]
        if len(hnr_values) > 0:
            features.extend([
                np.mean(hnr_values), np.std(hnr_values),
                np.min(hnr_values), np.max(hnr_values)
            ])
        else:
            features.extend([0.0] * 4)
        
        # ===== Temporal features (7 признаков) =====
        features.extend([
            np.sum(segment ** 2),
            np.sqrt(np.mean(segment ** 2)),
            np.sum(np.abs(np.diff(np.signbit(segment)))) / len(segment),
            np.max(np.abs(segment)),
            np.mean(np.abs(segment)),
            np.std(segment),
        ])
        
        # Signal entropy
        hist, _ = np.histogram(segment, bins=50, density=True)
        hist = hist[hist > 0]
        if len(hist) > 0:
            features.append(-np.sum(hist * np.log2(hist + 1e-10)))
        else:
            features.append(0.0)
        
        # Защита от несоответствия размерности
        if len(features) != 123:
            if len(features) < 123:
                features = np.pad(features, (0, 123 - len(features)), mode='constant', constant_values=0.0)
            else:
                features = features[:123]
    except Exception as e:
        print(f"⚠️ Ошибка при извлечении признаков: {e}")
        features = [0.0] * 123
    
    return np.array(features, dtype=np.float32)

# ==================== СЕГМЕНТАЦИЯ ====================
def split_audio_into_segments(filepath, segment_duration=SEGMENT_DURATION, sr_target=SAMPLE_RATE):
    y, sr = librosa.load(filepath, sr=sr_target)
    if y.ndim > 1:
        y = y[0]
    
    total_duration = len(y) / sr
    num_segments = int(total_duration // segment_duration)
    total_samples = num_segments * int(segment_duration * sr)
    y = y[:total_samples]
    
    segment_samples = int(segment_duration * sr)
    segments =[]
    for i in range(0, len(y), segment_samples):
        segment = y[i:i + segment_samples]
        if len(segment) == segment_samples:
            segments.append(segment)
    
    return segments

def extract_spectrogram_for_segment(segment, sr=SAMPLE_RATE, n_mels=N_MELS, 
                                    n_fft=N_FFT, hop_length=HOP_LENGTH):
    mel_spec = librosa.feature.melspectrogram(
        y=segment, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    if log_mel.shape[1] < TARGET_FRAMES:
        pad = TARGET_FRAMES - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0, 0), (0, pad)), mode='constant')
    else:
        log_mel = log_mel[:, :TARGET_FRAMES]
    
    log_mel = (log_mel - np.mean(log_mel)) / (np.std(log_mel) + 1e-8)
    return log_mel

# ==================== ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ====================
def preprocess_all_files_once(filepaths, labels, patient_ids, language_ids):
    print("\n" + "="*80)
    print("ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА (123 акустических признака)")
    print("="*80)
    
    segments_specs =[]
    segments_acoustics = []
    segments_labels = []
    segments_file_indices =[]
    segments_patient_ids = []
    segments_language_ids =[]
    
    original_file_labels = np.array(labels)
    original_file_paths = np.array(filepaths)
    original_patient_ids = np.array(patient_ids)
    original_language_ids = np.array(language_ids)
    
    total_segments = 0
    skipped_short = 0
    start_time = time.time()
    
    for file_idx, (filepath, label, pid, lid) in enumerate(zip(filepaths, labels, patient_ids, language_ids)):
        segments = split_audio_into_segments(filepath)
        if len(segments) == 0:
            skipped_short += 1
            continue
        
        for segment in segments:
            spec = extract_spectrogram_for_segment(segment)
            segments_specs.append(spec)
            
            acoustic_feat = extract_acoustic_features_for_segment(segment)
            segments_acoustics.append(acoustic_feat)
            
            segments_labels.append(label)
            segments_file_indices.append(file_idx)
            segments_patient_ids.append(pid)
            segments_language_ids.append(lid)
        
        total_segments += len(segments)
        
        if (file_idx + 1) % 10 == 0 or file_idx == len(filepaths) - 1:
            elapsed = time.time() - start_time
            eta = elapsed / (file_idx + 1) * (len(filepaths) - file_idx - 1) if file_idx > 0 else 0
            print(f"  Обработано {file_idx+1}/{len(filepaths)} файлов | "
                  f"Сегментов: {total_segments} | ETA: {eta:.1f} сек", end='\r')
    
    segments_specs = np.array(segments_specs, dtype=np.float32)
    segments_acoustics = np.array(segments_acoustics, dtype=np.float32)
    segments_labels = np.array(segments_labels, dtype=np.int64)
    segments_file_indices = np.array(segments_file_indices, dtype=np.int64)
    segments_patient_ids = np.array(segments_patient_ids, dtype=np.int64)
    segments_language_ids = np.array(segments_language_ids, dtype=np.int64)
    
    processing_time = time.time() - start_time
    print(f"\n✅ Предварительная обработка завершена за {processing_time:.2f} сек!")
    print(f"   Всего файлов: {len(filepaths)} (пропущено коротких: {skipped_short})")
    print(f"   Всего сегментов: {total_segments}")
    print(f"   Акустических признаков: {segments_acoustics.shape[1]}")
    
    # 🆕 Статистика по языкам
    print(f"\n📊 Распределение сегментов по языковым группам:")
    for lang_id in range(NUM_LANGUAGES):
        count = (segments_language_ids == lang_id).sum()
        print(f"   {LANG_NAMES[lang_id]}: {count} сегментов ({count/len(segments_language_ids)*100:.1f}%)")
    
    return {
        'segments_specs': segments_specs,
        'segments_acoustics': segments_acoustics,
        'segments_labels': segments_labels,
        'segments_file_indices': segments_file_indices,
        'segments_patient_ids': segments_patient_ids,
        'segments_language_ids': segments_language_ids,
        'original_file_labels': original_file_labels,
        'original_file_paths': original_file_paths,
        'original_patient_ids': original_patient_ids,
        'original_language_ids': original_language_ids
    }

# ==================== ДАТАСЕТ ====================
class PreprocessedSegmentDataset(Dataset):
    def __init__(self, segments_specs, segments_acoustics, segments_labels,
                 segments_file_indices, segments_patient_ids, segments_language_ids,
                 segment_mask=None, scaler=None):
        
        if segment_mask is not None:
            self.segments_specs = segments_specs[segment_mask]
            self.segments_acoustics_raw = segments_acoustics[segment_mask]
            self.segments_labels = segments_labels[segment_mask]
            self.segments_file_indices = segments_file_indices[segment_mask]
            self.segments_patient_ids = segments_patient_ids[segment_mask]
            self.segments_language_ids = segments_language_ids[segment_mask]
        else:
            self.segments_specs = segments_specs
            self.segments_acoustics_raw = segments_acoustics
            self.segments_labels = segments_labels
            self.segments_file_indices = segments_file_indices
            self.segments_patient_ids = segments_patient_ids
            self.segments_language_ids = segments_language_ids
        
        if scaler is None:
            self.scaler = StandardScaler()
            self.segments_acoustics = self.scaler.fit_transform(self.segments_acoustics_raw)
        else:
            self.scaler = scaler
            self.segments_acoustics = self.scaler.transform(self.segments_acoustics_raw)
    
    def __len__(self):
        return len(self.segments_specs)
    
    def __getitem__(self, idx):
        return (
            self.segments_specs[idx].copy(),
            self.segments_acoustics[idx].copy(),
            self.segments_labels[idx],
            self.segments_file_indices[idx],
            self.segments_language_ids[idx]
        )

def collate_fn(batch):
    specs = torch.from_numpy(np.stack([item[0] for item in batch], axis=0)).unsqueeze(1)
    acoustics = torch.from_numpy(np.stack([item[1] for item in batch], axis=0))
    labels = torch.tensor([item[2] for item in batch], dtype=torch.long)
    file_indices = torch.tensor([item[3] for item in batch], dtype=torch.long)
    lang_ids = torch.tensor([item[4] for item in batch], dtype=torch.long)
    return specs, acoustics, labels, file_indices, lang_ids

# ==================== МОДЕЛЬ ====================
class HybridModel(nn.Module):
    def __init__(self, acoustic_feature_size=123, n_mels=128, target_length=437):
        super(HybridModel, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
            
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
            
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
            
            nn.Conv2d(128, 256, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        self.rnn = nn.LSTM(
            input_size=256 * 4,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        self.mkl = nn.Sequential(
            nn.Linear(128 * 2 + 256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.mlp = nn.Sequential(
            nn.Linear(512 + acoustic_feature_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.classifier = nn.Linear(128, 2)
    
    def forward(self, x_spectrogram, x_acoustic):
        cnn_out = self.cnn(x_spectrogram)
        rnn_input = cnn_out.permute(0, 3, 1, 2).contiguous().view(cnn_out.size(0), cnn_out.size(3), -1)
        rnn_out, _ = self.rnn(rnn_input)
        rnn_last = rnn_out[:, -1, :]
        cnn_flat = cnn_out.view(cnn_out.size(0), -1)
        fused = torch.cat([rnn_last, cnn_flat], dim=1)
        mkl_out = self.mkl(fused)
        combined = torch.cat([mkl_out, x_acoustic], dim=1)
        mlp_out = self.mlp(combined)
        return self.classifier(mlp_out)

# ==================== 🆕 ВАЛИДАЦИЯ С CONFUSION MATRIX ПО ЯЗЫКАМ ====================
def validate_file_level(model, dataset, device=DEVICE, batch_size=32):
    model.eval()
    
    all_specs = torch.from_numpy(dataset.segments_specs).unsqueeze(1).to(device, non_blocking=True)
    all_acoustics = torch.from_numpy(dataset.segments_acoustics).to(device, non_blocking=True)
    all_labels = torch.from_numpy(dataset.segments_labels).to(device, non_blocking=True)
    all_file_indices = torch.from_numpy(dataset.segments_file_indices).to(device, non_blocking=True)
    all_lang_ids = torch.from_numpy(dataset.segments_language_ids).to(device, non_blocking=True)
    
    all_logits =[]
    with torch.no_grad():
        for i in range(0, len(all_specs), batch_size):
            batch_specs = all_specs[i:i+batch_size]
            batch_acoustics = all_acoustics[i:i+batch_size]
            logits = model(batch_specs, batch_acoustics)
            all_logits.append(logits)
    
    all_logits = torch.cat(all_logits, dim=0).cpu()
    all_labels = all_labels.cpu()
    all_file_indices = all_file_indices.cpu()
    all_lang_ids = all_lang_ids.cpu()
    
    unique_file_indices = torch.unique(all_file_indices)
    file_predictions =[]
    file_true_labels = []
    file_probabilities = []
    file_language_ids =[]
    
    for file_idx in unique_file_indices:
        mask = (all_file_indices == file_idx)
        file_logits = all_logits[mask]
        file_label = all_labels[mask][0].item()
        file_lang_id = all_lang_ids[mask][0].item()
        
        probs = torch.softmax(file_logits, dim=1)
        mean_probs = probs.mean(dim=0)
        pred_class = mean_probs.argmax().item()
        
        file_predictions.append(pred_class)
        file_true_labels.append(file_label)
        file_probabilities.append(mean_probs[1].item())
        file_language_ids.append(file_lang_id)
    
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    
    # 🆕 Confusion matrix по языковым группам
    confusion_by_language = {}
    for lang_id in range(NUM_LANGUAGES):
        lang_mask = np.array(file_language_ids) == lang_id
        if lang_mask.sum() > 0:
            lang_true = np.array(file_true_labels)[lang_mask]
            lang_pred = np.array(file_predictions)[lang_mask]
            confusion_by_language[lang_id] = confusion_matrix(lang_true, lang_pred)
    
    return {
        'accuracy': accuracy_score(file_true_labels, file_predictions),
        'precision': precision_score(file_true_labels, file_predictions, zero_division=0),
        'recall': recall_score(file_true_labels, file_predictions, zero_division=0),
        'f1': f1_score(file_true_labels, file_predictions, zero_division=0),
        'roc_auc': roc_auc_score(file_true_labels, file_probabilities) if len(set(file_true_labels)) > 1 else 0.0,
        'confusion_matrix': confusion_matrix(file_true_labels, file_predictions),
        'confusion_by_language': confusion_by_language,  # 🆕
        'predictions': file_predictions,
        'true_labels': file_true_labels,
        'probabilities': file_probabilities,
        'language_ids': file_language_ids  # 🆕
    }

# ==================== CUTMIX ====================
def cutmix_spectrograms(spec, spec2, beta=CUTMIX_BETA):
    batch_size, channels, h, w = spec.shape
    
    cut_ratio = np.sqrt(1.0 - np.random.beta(beta, beta))
    cut_h = int(h * cut_ratio)
    cut_w = int(w * cut_ratio)
    
    cx = np.random.randint(0, w)
    cy = np.random.randint(0, h)
    
    x1 = max(0, cx - cut_w // 2)
    x2 = min(w, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    y2 = min(h, cy + cut_h // 2)
    
    mask = torch.ones((batch_size, channels, h, w), device=spec.device)
    mask[:, :, y1:y2, x1:x2] = 0.0
    
    spec_cutmix = spec.clone()
    spec_cutmix[:, :, y1:y2, x1:x2] = spec2[:, :, y1:y2, x1:x2]
    
    cut_area = (x2 - x1) * (y2 - y1)
    total_area = w * h
    cut_ratio_actual = cut_area / total_area
    
    return spec_cutmix, mask, cut_ratio_actual

# ==================== СОХРАНЕНИЕ МОДЕЛИ ====================
def save_best_model(model_state, scaler, metrics, fold_num, hyperparameters, 
                   filepath_template="parkinson_best_fold_{fold_num}.pt"):
    save_path = filepath_template.format(fold_num=fold_num)
    
    checkpoint = {
        'model_state_dict': model_state,
        'scaler': scaler,
        'metrics': {
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'roc_auc': metrics['roc_auc'],
            'confusion_matrix': metrics['confusion_matrix'].tolist(),
            'fold_num': fold_num
        },
        'hyperparameters': hyperparameters,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'architecture': 'Hybrid CNN-LSTM with acoustic features (123-dim)'
    }
    
    torch.save(checkpoint, save_path)
    print(f"✅ Лучшая модель фолда {fold_num} сохранена: {save_path}")
    
    metadata_path = save_path.replace('.pt', '_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump({
            'metrics': checkpoint['metrics'],
            'hyperparameters': checkpoint['hyperparameters'],
            'timestamp': checkpoint['timestamp']
        }, f, indent=2, ensure_ascii=False)
    print(f"📄 Метаданные сохранены: {metadata_path}")
    
    return save_path

# ==================== ЗАГРУЗКА МОДЕЛИ ====================
def load_model(checkpoint_path, device=torch.device('cpu')):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model = HybridModel(
        acoustic_feature_size=123,
        n_mels=128,
        target_length=TARGET_FRAMES
    ).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    scaler = checkpoint['scaler']
    metrics = checkpoint['metrics']
    
    print(f"✅ Загружена модель: Acc={metrics['accuracy']:.4f}, F1={metrics['f1']:.4f}, AUC={metrics['roc_auc']:.4f}")
    return model, scaler, metrics

# ==================== ОБУЧЕНИЕ ====================
def train_with_group_cv(preprocessed_data, n_splits=N_SPLITS, random_state=42):
    segments_specs = preprocessed_data['segments_specs']
    segments_acoustics = preprocessed_data['segments_acoustics']
    segments_labels = preprocessed_data['segments_labels']
    segments_file_indices = preprocessed_data['segments_file_indices']
    segments_patient_ids = preprocessed_data['segments_patient_ids']
    segments_language_ids = preprocessed_data['segments_language_ids']
    original_patient_ids = preprocessed_data['original_patient_ids']
    
    print(f"\n📊 Данные для обучения:")
    print(f"   Всего пациентов: {len(np.unique(original_patient_ids))}")
    print(f"   Всего файлов: {len(np.unique(segments_file_indices))}")
    print(f"   Всего сегментов: {len(segments_specs)}")
    print(f"   Акустических признаков: {segments_acoustics.shape[1]}")
    
    print(f"\n🔄 Групповая кросс-валидация: {n_splits} фолдов")
    
    gkf = GroupKFold(n_splits=n_splits)
    fold_results =[]
    
    unique_file_indices = np.unique(segments_file_indices)
    file_labels_for_cv = segments_labels[np.searchsorted(segments_file_indices, unique_file_indices)]
    patient_groups_for_cv = original_patient_ids[unique_file_indices]
    
    hyperparameters = {
        'segment_duration': SEGMENT_DURATION,
        'sample_rate': SAMPLE_RATE,
        'n_mels': N_MELS,
        'acoustic_features': 123,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'cutmix_enabled': USE_CUTMIX,
        'cutmix_prob': CUTMIX_PROB if USE_CUTMIX else 0.0,
        'num_epochs': NUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE
    }
    
    fold_counter = 0
    for train_file_idx, val_file_idx in gkf.split(unique_file_indices, file_labels_for_cv, 
                                                   groups=patient_groups_for_cv):
        fold_counter += 1
        print(f"\n{'='*80}")
        print(f"НАЧАЛО ОБУЧЕНИЯ ФОЛДА {fold_counter}/{n_splits} на {DEVICE}")
        print(f"{'='*80}")
        
        train_files = unique_file_indices[train_file_idx]
        val_files = unique_file_indices[val_file_idx]
        
        train_patients = np.unique(original_patient_ids[train_file_idx])
        val_patients = np.unique(original_patient_ids[val_file_idx])
        
        intersection = set(train_patients) & set(val_patients)
        if intersection:
            print(f"⚠️ ВНИМАНИЕ: пересечение пациентов: {intersection}")
        else:
            print(f"✅ Проверка пройдена: нет пересечения пациентов")
        
        train_segment_mask = np.isin(segments_file_indices, train_files)
        val_segment_mask = np.isin(segments_file_indices, val_files)
        
        print(f"Train: {len(train_files)} файлов ({len(train_patients)} пациентов) → {train_segment_mask.sum()} сегментов")
        print(f"Val:   {len(val_files)} файлов ({len(val_patients)} пациентов) → {val_segment_mask.sum()} сегментов")
        
        train_dataset = PreprocessedSegmentDataset(
            segments_specs, segments_acoustics, segments_labels,
            segments_file_indices, segments_patient_ids, segments_language_ids,
            train_segment_mask
        )
        
        val_dataset = PreprocessedSegmentDataset(
            segments_specs, segments_acoustics, segments_labels,
            segments_file_indices, segments_patient_ids, segments_language_ids,
            val_segment_mask,
            scaler=train_dataset.scaler
        )
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            collate_fn=collate_fn
        )
        
        model = HybridModel(
            acoustic_feature_size=123,
            n_mels=128,
            target_length=TARGET_FRAMES
        ).to(DEVICE)
        
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )
        
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=5
        )
        
        best_val_file_acc = 0.0
        patience_counter = 0
        history = {
            'train_loss':[], 'train_acc_seg': [],
            'val_acc_file':[], 'val_f1_file': [], 'val_auc_file':[]
        }
        
        for epoch in range(NUM_EPOCHS):
            epoch_start = time.time()
            model.train()
            
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            cutmix_applied = 0
            
            for spec, acoustic, labels_batch, _, _ in train_loader:
                spec = spec.to(DEVICE, non_blocking=PIN_MEMORY)
                acoustic = acoustic.to(DEVICE, non_blocking=PIN_MEMORY)
                labels_batch = labels_batch.to(DEVICE, non_blocking=PIN_MEMORY)
                
                optimizer.zero_grad()
                
                if USE_CUTMIX and random.random() < CUTMIX_PROB:
                    indices = torch.randperm(spec.size(0))
                    spec2 = spec[indices]
                    labels2 = labels_batch[indices]
                    
                    spec_cutmix, mask, cut_ratio = cutmix_spectrograms(spec, spec2, beta=CUTMIX_BETA)
                    
                    if cut_ratio < 0.5:
                        dominant_labels = labels_batch
                    else:
                        dominant_labels = labels2
                    
                    logits = model(spec_cutmix, acoustic)
                    loss = criterion(logits, dominant_labels)
                    
                    _, predicted = logits.max(1)
                    train_correct += predicted.eq(dominant_labels).sum().item()
                    cutmix_applied += 1
                else:
                    logits = model(spec, acoustic)
                    loss = criterion(logits, labels_batch)
                    
                    _, predicted = logits.max(1)
                    train_correct += predicted.eq(labels_batch).sum().item()
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                train_loss += loss.item() * labels_batch.size(0)
                train_total += labels_batch.size(0)
            
            train_loss_avg = train_loss / train_total
            train_acc_seg = train_correct / train_total
            
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
            
            val_file_metrics = validate_file_level(model, val_dataset, device=DEVICE, batch_size=BATCH_SIZE*2)
            val_acc_file = val_file_metrics['accuracy']
            val_f1_file = val_file_metrics['f1']
            val_auc_file = val_file_metrics['roc_auc']
            
            history['train_loss'].append(train_loss_avg)
            history['train_acc_seg'].append(train_acc_seg)
            history['val_acc_file'].append(val_acc_file)
            history['val_f1_file'].append(val_f1_file)
            history['val_auc_file'].append(val_auc_file)
            
            epoch_time = time.time() - epoch_start
            lr = optimizer.param_groups[0]['lr']
            
            if val_acc_file > best_val_file_acc + IMPROVEMENT_THRESHOLD:
                best_val_file_acc = val_acc_file
                best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
                best_metrics_file = val_file_metrics
                best_scaler = train_dataset.scaler
                patience_counter = 0
                improvement_status = "🏆 УЛУЧШЕНИЕ"
            else:
                patience_counter += 1
                improvement_status = "⏳ Без улучшения"
            
            scheduler.step(val_acc_file)
            
            if (epoch + 1) % 10 == 0 or epoch == 0 or patience_counter == 0:
                print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
                      f"LR: {lr:.1e} | "
                      f"CutMix: {cutmix_applied/len(train_loader)*100:.1f}% | "
                      f"Train Loss: {train_loss_avg:.4f} | "
                      f"Train Acc (seg): {train_acc_seg:.4f} | "
                      f"Val Acc (file): {val_acc_file:.4f} | "
                      f"Val F1 (file): {val_f1_file:.4f} | "
                      f"{improvement_status} | Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE} | "
                      f"Time: {epoch_time:.1f}s")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️ Ранняя остановка на эпохе {epoch+1} "
                      f"(лучшая точность файла: {best_val_file_acc:.4f})")
                break
        
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_model_state.items()})
        
        model_path = save_best_model(
            best_model_state,
            best_scaler,
            best_metrics_file,
            fold_counter,
            hyperparameters,
            filepath_template="parkinson_best_fold_{fold_num}.pt"
        )
        
        fold_results.append({
            'fold_num': fold_counter,
            'best_val_acc_file': best_metrics_file['accuracy'],
            'best_val_f1_file': best_metrics_file['f1'],
            'best_val_auc_file': best_metrics_file['roc_auc'],
            'confusion_matrix_file': best_metrics_file['confusion_matrix'],
            'confusion_by_language': best_metrics_file['confusion_by_language'],  # 🆕
            'history': history,
            'train_patients': len(train_patients),
            'val_patients': len(val_patients),
            'model_path': model_path
        })
        
        print(f"✅ Фолд {fold_counter} завершен. Лучшая точность (файл): {best_val_file_acc:.4f}")
    
    return fold_results

# ==================== ГЛОБАЛЬНО ЛУЧШАЯ МОДЕЛЬ ====================
def select_and_save_global_best(fold_results, output_path="parkinson_global_best.pt"):
    best_fold = max(fold_results, key=lambda x: x['best_val_acc_file'])
    
    print("\n" + "="*80)
    print("🏆 ГЛОБАЛЬНО ЛУЧШАЯ МОДЕЛЬ")
    print("="*80)
    print(f"Фолд: {best_fold['fold_num']}")
    print(f"Точность: {best_fold['best_val_acc_file']:.4f}")
    print(f"F1-score: {best_fold['best_val_f1_file']:.4f}")
    print(f"ROC AUC: {best_fold['best_val_auc_file']:.4f}")
    
    shutil.copy2(best_fold['model_path'], output_path)
    print(f"\n✅ Глобально лучшая модель: {output_path}")
    
    return best_fold, output_path

# ==================== 🆕 ВИЗУАЛИЗАЦИЯ С CONFUSION MATRIX ПО ЯЗЫКАМ ====================
def plot_training_history(history, fold_num):
    plt.figure(figsize=(18, 5))
    
    plt.subplot(1, 4, 1)
    plt.plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
    plt.title(f'Fold {fold_num}: Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.subplot(1, 4, 2)
    plt.plot(history['train_acc_seg'], label='Train Acc (seg)', marker='o', linewidth=2)
    plt.plot(history['val_acc_file'], label='Val Acc (file)', marker='s', linewidth=2)
    plt.title(f'Fold {fold_num}: Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.subplot(1, 4, 3)
    plt.plot(history['val_f1_file'], label='Val F1 (file)', marker='^', linewidth=2, color='green')
    plt.title(f'Fold {fold_num}: F1-Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1-Score')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.subplot(1, 4, 4)
    plt.plot(history['val_auc_file'], label='Val AUC (file)', marker='d', linewidth=2, color='purple')
    plt.title(f'Fold {fold_num}: ROC AUC')
    plt.xlabel('Epoch')
    plt.ylabel('ROC AUC')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(f'training_history_fold_{fold_num}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📊 Графики обучения: training_history_fold_{fold_num}.png")

def summarize_cv_results(fold_results):
    print("\n" + "="*80)
    print(f"ИТОГОВЫЕ РЕЗУЛЬТАТЫ {N_SPLITS}-FOLD КРОСС-ВАЛИДАЦИИ")
    print("="*80)
    
    accs_file = [r['best_val_acc_file'] for r in fold_results]
    f1s_file = [r['best_val_f1_file'] for r in fold_results]
    aucs_file = [r['best_val_auc_file'] for r in fold_results]
    
    print(f"\n📊 Метрики на уровне ФАЙЛОВ:")
    print("-"*80)
    print(f"  Среднее: Acc={np.mean(accs_file):.4f} ± {np.std(accs_file):.4f} | "
          f"F1={np.mean(f1s_file):.4f} ± {np.std(f1s_file):.4f} | "
          f"AUC={np.mean(aucs_file):.4f} ± {np.std(aucs_file):.4f}")
    
    # 🆕 Confusion matrix по языковым группам
    print(f"\n📊 Confusion Matrix по языковым группам:")
    print("-"*80)
    
    # Усреднённая confusion matrix по всем языкам
    avg_cm_by_lang = {lang_id:[] for lang_id in range(NUM_LANGUAGES)}
    for r in fold_results:
        if 'confusion_by_language' in r:
            for lang_id, cm in r['confusion_by_language'].items():
                avg_cm_by_lang[lang_id].append(cm)
    
    # 🆕 Визуализация confusion matrix для каждой языковой группы
    fig, axes = plt.subplots(1, NUM_LANGUAGES + 1, figsize=(6*(NUM_LANGUAGES+1), 5))
    
    # Общая confusion matrix
    avg_cm = np.zeros((2, 2))
    for r in fold_results:
        avg_cm += r['confusion_matrix_file']
    avg_cm = avg_cm / len(fold_results)
    
    sns.heatmap(avg_cm, annot=True, fmt='.1f', cmap='Blues', ax=axes[0],
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    axes[0].set_title('Общая (все языки)')
    axes[0].set_ylabel('Истинный класс')
    axes[0].set_xlabel('Предсказанный класс')
    
    # 🆕 Confusion matrix для каждой языковой группы
    for lang_id in range(NUM_LANGUAGES):
        if avg_cm_by_lang[lang_id]:
            lang_cm = np.mean(avg_cm_by_lang[lang_id], axis=0)
            sns.heatmap(lang_cm, annot=True, fmt='.1f', cmap='Blues', ax=axes[lang_id+1],
                        xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
            axes[lang_id+1].set_title(f'{LANG_NAMES[lang_id]}')
            axes[lang_id+1].set_ylabel('Истинный класс')
            axes[lang_id+1].set_xlabel('Предсказанный класс')
        else:
            axes[lang_id+1].text(0.5, 0.5, f'Нет данных\nдля {LANG_NAMES[lang_id]}', 
                                ha='center', va='center', fontsize=12)
            axes[lang_id+1].set_title(f'{LANG_NAMES[lang_id]}')
    
    plt.tight_layout()
    plt.savefig('confusion_matrix_by_language.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("📊 Confusion matrix по языкам: confusion_matrix_by_language.png")
    
    # Распределение метрик
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.hist(accs_file, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
    plt.axvline(np.mean(accs_file), color='red', linestyle='--', linewidth=2, 
                label=f'Среднее: {np.mean(accs_file):.4f}')
    plt.title('Распределение точности')
    plt.xlabel('Accuracy')
    plt.ylabel('Частота')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 2)
    plt.hist(f1s_file, bins=10, color='lightgreen', edgecolor='black', alpha=0.7)
    plt.axvline(np.mean(f1s_file), color='red', linestyle='--', linewidth=2,
                label=f'Среднее: {np.mean(f1s_file):.4f}')
    plt.title('Распределение F1-score')
    plt.xlabel('F1 Score')
    plt.ylabel('Частота')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 3)
    plt.hist(aucs_file, bins=10, color='lightcoral', edgecolor='black', alpha=0.7)
    plt.axvline(np.mean(aucs_file), color='red', linestyle='--', linewidth=2,
                label=f'Среднее: {np.mean(aucs_file):.4f}')
    plt.title('Распределение ROC AUC')
    plt.xlabel('ROC AUC')
    plt.ylabel('Частота')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cv_metrics_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("📊 Распределение метрик: cv_metrics_distribution.png")
    
    # Усреднённая матрица ошибок (общая)
    plt.figure(figsize=(6, 5))
    sns.heatmap(avg_cm, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    plt.title('Усреднённая матрица ошибок (все языки)')
    plt.ylabel('Истинный класс')
    plt.xlabel('Предсказанный класс')
    plt.tight_layout()
    plt.savefig('confusion_matrix_avg.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("📊 Матрица ошибок: confusion_matrix_avg.png")
    
    print("\n" + "="*80)
    print("ДЕТАЛИЗИРОВАННЫЕ РЕЗУЛЬТАТЫ ПО ФОЛДАМ")
    print("="*80)
    print(f"{'Фолд':<6} {'Точность':<12} {'F1-score':<12} {'ROC AUC':<12} {'Путь к модели'}")
    print("-"*80)
    for r in fold_results:
        print(f"{r['fold_num']:<6} {r['best_val_acc_file']:<12.4f} {r['best_val_f1_file']:<12.4f} "
              f"{r['best_val_auc_file']:<12.4f} {r['model_path']}")
    print("="*80)

# ==================== ЗАПУСК ====================
if __name__ == "__main__":
    filepaths = df["filepath"].tolist()
    labels = (df["label"] == "PD").astype(int).tolist()
    patient_ids_list = df["patient_id"].tolist()
    language_ids_list = df["language_id"].tolist()
    
    preprocessed_data = preprocess_all_files_once(
        filepaths, labels, patient_ids_list, language_ids_list
    )
    
    actual_feature_dim = preprocessed_data['segments_acoustics'].shape[1]
    print(f"\n🔍 Проверка размерности: {actual_feature_dim} признаков")
    if actual_feature_dim != 123:
        print(f"⚠️ ВНИМАНИЕ: ожидаемая размерность 123, получено {actual_feature_dim}")
    
    print("\n" + "="*80)
    print(f"НАЧАЛО ОБУЧЕНИЯ С {N_SPLITS}-FOLD КРОСС-ВАЛИДАЦИЕЙ")
    print("="*80)
    print("🎯 КЛЮЧЕВЫЕ ОСОБЕННОСТИ:")
    print("   • 123 акустических признака")
    print("   • GroupKFold — разделение по пациентам")
    print("   • CutMix для спектрограмм")
    print("   • 🆕 Confusion Matrix по языковым группам")
    
    fold_results = train_with_group_cv(
        preprocessed_data,
        n_splits=N_SPLITS,
        random_state=42
    )
    
    for r in fold_results:
        plot_training_history(r['history'], r['fold_num'])
    
    summarize_cv_results(fold_results)
    
    best_fold, global_best_path = select_and_save_global_best(
        fold_results, "parkinson_global_best.pt"
    )
    
    print("\n✅ Обучение завершено успешно!")
    print(f"\n📌 СОХРАНЁННЫЕ ФАЙЛЫ:")
    print(f"   • Модели фолдов: parkinson_best_fold_*.pt")
    print(f"   • Глобально лучшая: {global_best_path}")
    print(f"   • 🆕 Confusion matrix по языкам: confusion_matrix_by_language.png")
    print(f"   • Графики: training_history_fold_*.png, cv_metrics_distribution.png")